# CELL 1 - INSTALL DEPENDENCIES

In [ ]:
!pip install -q transformers peft accelerate bitsandbytes huggingface_hub

# CELL 2 - MOUNT GOOGLE DRIVE


In [ ]:
from google.colab import drive
drive.mount('your-drive-here')
# the model will be avaliable for you to download in github

# CELL 3 - IMPORT LIBRARIES


In [ ]:
import os
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
from huggingface_hub import login

# CELL 4 - HUGGING FACE AUTHENTICATION


In [ ]:
import getpass
HF_TOKEN = getpass.getpass("Enter your Hugging Face token: ")
os.environ["HF_TOKEN"] = HF_TOKEN

# CELL 5 - VISABUDDY CLASS DEFINITION


In [ ]:
# IMPROVE VISA BUDDY WITH SYSTEM PROMPT
class ImprovedVisaBuddy:
    def __init__(self, model_path="path to your model here"):
        self.model_path = model_path
        self.model = None
        self.tokenizer = None
        self.load_model()

    def load_model(self):
        print("🔄 Loading Improved Visa Buddy model...")

        base_model = "meta-llama/Llama-3.1-8B-Instruct"

        try:
            self.tokenizer = AutoTokenizer.from_pretrained(
                base_model,
                token=os.environ["HF_TOKEN"]
            )
            if self.tokenizer.pad_token_id is None:
                self.tokenizer.pad_token_id = self.tokenizer.eos_token_id

            self.model = AutoModelForCausalLM.from_pretrained(
                base_model,
                torch_dtype=torch.float16,
                device_map="auto",
                load_in_8bit=True,
                token=os.environ["HF_TOKEN"]
            )

            if os.path.exists(self.model_path):
                self.model = PeftModel.from_pretrained(self.model, self.model_path)
                print("✅ Fine-tuned adapter loaded!")
            else:
                print("⚠️  Using base model (no fine-tuned adapter found)")

            self.model.eval()
            print("✅ Improved Visa Buddy model loaded successfully!")

        except Exception as e:
            print(f"❌ Error loading model: {e}")

    def chat(self, message, max_length=500):
        if self.model is None:
            return "Model not loaded."

        # SYSTEM PROMPT - This is the key improvement!
        system_prompt = """You are Visa Buddy, a specialized AI assistant for Canadian immigration and visa information.

YOUR ROLE:
- Provide clear, direct, and accurate information about Canadian visas, immigration, and related processes
- Focus exclusively on Canadian immigration topics
- Be concise and factual - avoid unnecessary fluff or conversational filler
- If asked about non-visa topics, politely redirect to Canadian immigration
- Use bullet points and clear structure when listing requirements
- Always cite specific programs, requirements, and processing times when possible

RESPONSE STYLE:
- Direct and professional
- Structured and easy to read
- Factual and up-to-date
- Focused on actionable information

NEVER:
- Engage in general chit-chat
- Provide opinions or personal advice
- Discuss topics unrelated to Canadian immigration
- Use vague or uncertain language"""

        # Format the prompt with system message
        prompt = f"""<|start_header_id|>system<|end_header_id|>

{system_prompt}<|eot_id|>
<|start_header_id|>user<|end_header_id|>

{message}<|eot_id|>
<|start_header_id|>assistant<|end_header_id|>

"""

        inputs = self.tokenizer(prompt, return_tensors="pt").to(self.model.device)

        with torch.no_grad():
            outputs = self.model.generate(
                **inputs,
                max_new_tokens=max_length,
                temperature=0.7,
                do_sample=True,
                pad_token_id=self.tokenizer.eos_token_id,
                eos_token_id=self.tokenizer.eos_token_id,
                repetition_penalty=1.1,
                no_repeat_ngram_size=3
            )

        response = self.tokenizer.decode(outputs[0], skip_special_tokens=True)

        # Extract only the assistant's response
        if "assistant" in response:
            response = response.split("assistant")[-1].strip()

        return response

# Replace the existing visa_buddy with improved version
visa_buddy = ImprovedVisaBuddy()
visa_buddy_instance = visa_buddy
print("🎯 Improved Visa Buddy with system prompt is ready!")

# CELL 6 - INITIALIZE MODEL


In [ ]:
visa_buddy_instance = visa_buddy


# CELL 7 - INSTALL DEPENDENCIES


In [ ]:
!pip install -q fastapi uvicorn python-multipart
!npm install -g localtunnel

# CELL 8 - FASTAPI SERVER


In [ ]:
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
import uvicorn
import threading

class ChatRequest(BaseModel):
    message: str
    max_length: int = 700

class ChatResponse(BaseModel):
    response: str

app = FastAPI(title="Visa Buddy API", version="1.0.0")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

visa_buddy_instance = None

@app.get("/")
async def root():
    return {"message": "Visa Buddy API is running!"}

@app.post("/chat", response_model=ChatResponse)
async def chat_endpoint(request: ChatRequest):
    try:
        if visa_buddy_instance is None:
            raise HTTPException(status_code=500, detail="Model not loaded")

        response = visa_buddy_instance.chat(request.message, request.max_length)
        return ChatResponse(response=response)
    except Exception as e:
        raise HTTPException(status_code=500, detail=f"Error: {str(e)}")

@app.get("/health")
async def health_check():
    return {"status": "healthy", "model_loaded": visa_buddy_instance is not None}

def start_server():
    uvicorn.run(app, host="0.0.0.0", port=8000)

visa_buddy_instance = visa_buddy
print("✅ Visa Buddy model ready for API!")

# CELL 9 - START SERVER & GET PUBLIC URL


In [ ]:
import threading
import time

# Start FastAPI server in background
server_thread = threading.Thread(target=start_server, daemon=True)
server_thread.start()
print("🔄 Starting FastAPI server...")
time.sleep(3)

print("🚀 Run this command in a NEW cell to get your public URL:")
print("!lt --port 8000 --subdomain visa-buddy-$(date +%s)")
print("\nOr use this simpler version:")
print("!lt --port 8000")
print("\n📝 Copy the URL it provides and use it in your React app!")

In [ ]:
!wget https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared-linux-amd64
!./cloudflared-linux-amd64 tunnel --url http://localhost:8000